# 37 — OpenAlex Career Age Pipeline
Fetches author profiles for ICWSM/JCDL award papers, computes career age, flags outliers, and saves a clean version for downstream analysis.

In [ ]:
import pandas as pd
import requests
import time

EMAIL = "your@email.com"  # replace — puts you in polite pool
HEADERS = {"User-Agent": f"thesis-research/1.0 (mailto:{EMAIL})"}
SLEEP = 0.12  # ~8 req/s, safely under 10 req/s free tier limit

def openalex_get(url, params=None):
    r = requests.get(url, headers=HEADERS, params=params)
    time.sleep(SLEEP)
    r.raise_for_status()
    return r.json()

In [ ]:
raw = pd.read_csv("../data/raw/icwsm_jcdl_awards_raw.csv")
matched = raw[raw["openalex_id"].notna()].copy()
print(f"Matched papers: {len(matched)}")
print(matched.groupby(["conference", "award_type"]).size())

## Step 1 — Fetch authorships per paper

In [ ]:
rows = []
for _, paper in matched.iterrows():
    oa_id = paper["openalex_id"].split("/")[-1]
    try:
        data = openalex_get(f"https://api.openalex.org/works/{oa_id}")
    except Exception as e:
        print(f"Error {oa_id}: {e}")
        continue

    authors = data.get("authorships", [])
    n = len(authors)
    for i, a in enumerate(authors):
        if i == 0:
            pos = "first"
        elif i == n - 1:
            pos = "last"
        else:
            pos = "middle"

        author_id = a.get("author", {}).get("id", None)
        author_name = a.get("author", {}).get("display_name", None)

        rows.append({
            "openalex_paper_id": paper["openalex_id"],
            "paper_title": paper["title"],
            "conference": paper["conference"],
            "award_type": paper["award_type"],
            "award_year": paper["year"],
            "author_id": author_id,
            "author_name": author_name,
            "author_position": pos,
        })

authorships = pd.DataFrame(rows)
print(f"Total authorship rows: {len(authorships)}")
authorships.head()

## Step 2 — Fetch author profiles (first_pub_year, total_works, total_cites)

In [ ]:
unique_authors = authorships[authorships["author_id"].notna()]["author_id"].unique()
print(f"Unique authors to fetch: {len(unique_authors)}")

author_profiles = {}
for aid in unique_authors:
    oa_id = aid.split("/")[-1]
    try:
        data = openalex_get(f"https://api.openalex.org/authors/{oa_id}")
    except Exception as e:
        print(f"Error {oa_id}: {e}")
        author_profiles[aid] = {"first_pub_year": None, "total_works": None, "total_cites": None}
        continue

    counts = data.get("counts_by_year", [])
    years_with_works = [c["year"] for c in counts if c.get("works_count", 0) > 0]
    first_pub_year = min(years_with_works) if years_with_works else None

    author_profiles[aid] = {
        "first_pub_year": first_pub_year,
        "total_works": data.get("works_count"),
        "total_cites": data.get("cited_by_count"),
    }

print(f"Profiles fetched: {len(author_profiles)}")

## Step 3 — Merge, compute career_age, is_junior

In [ ]:
profile_df = pd.DataFrame.from_dict(author_profiles, orient="index").reset_index()
profile_df.columns = ["author_id", "first_pub_year", "total_works", "total_cites"]

df = authorships.merge(profile_df, on="author_id", how="left")

df["career_age_missing"] = df["first_pub_year"].isna()
df["career_age"] = df["award_year"] - df["first_pub_year"]
df["is_junior"] = df["career_age"] < 5

print(df[["author_name", "career_age", "is_junior"]].describe())
df.to_csv("../data/raw/icwsm_jcdl_author_profiles.csv", index=False)
print("Saved → data/raw/icwsm_jcdl_author_profiles.csv")

## Step 4 — Outlier Detection & Cleaning

OpenAlex `counts_by_year` only goes back ~10–15 years in its index, so very old or very famous researchers sometimes get a `first_pub_year` that is actually their **birth year** or a data artifact (e.g. 1674, 1800, 1852).  
Rules for flagging:
- `first_pub_year < 1940` → almost certainly not a real publication year for a living author
- `career_age < 0` → first_pub_year is *after* award_year (data lag / pre-print artifact)
- `career_age_missing == True` → no profile data at all

In [ ]:
df = pd.read_csv("../data/raw/icwsm_jcdl_author_profiles.csv")

CUTOFF_YEAR = 1940

df["outlier_ancient"] = df["first_pub_year"] < CUTOFF_YEAR
df["outlier_negative"] = df["career_age"] < 0
df["career_age_valid"] = (
    ~df["career_age_missing"] &
    ~df["outlier_ancient"] &
    ~df["outlier_negative"]
)

flagged = df[df["outlier_ancient"] | df["outlier_negative"] | df["career_age_missing"]]

print(f"Total rows         : {len(df)}")
print(f"  career_age_missing : {df['career_age_missing'].sum()}")
print(f"  outlier_ancient    : {df['outlier_ancient'].sum()}  (first_pub_year < {CUTOFF_YEAR})")
print(f"  outlier_negative   : {df['outlier_negative'].sum()}  (career_age < 0)")
print(f"  career_age_valid   : {df['career_age_valid'].sum()} rows usable")

print("\n--- Flagged rows ---")
display(
    flagged[["author_name", "paper_title", "conference", "award_year",
             "first_pub_year", "career_age", "outlier_ancient", "outlier_negative", "career_age_missing"]]
    .sort_values("first_pub_year")
    .reset_index(drop=True)
)

In [ ]:
df.to_csv("../data/raw/icwsm_jcdl_author_profiles_clean.csv", index=False)
print("Saved → data/raw/icwsm_jcdl_author_profiles_clean.csv")
print("Use career_age_valid == True to filter to reliable rows in downstream analysis.")

## Step 5 — EDA on valid rows

In [ ]:
import matplotlib.pyplot as plt

valid = df[df["career_age_valid"]].copy()
print(f"Valid rows: {len(valid)} | Unique authors: {valid['author_id'].nunique()}")

# Overall junior share
j = valid["is_junior"].value_counts(normalize=True).rename({True: "Junior (<5yr)", False: "Senior (≥5yr)"})
print("\nOverall junior/senior split:")
print(j.round(3))

# By conference
print("\nJunior share by conference:")
print(valid.groupby("conference")["is_junior"].mean().round(3))

# By award type
print("\nJunior share by award type:")
print(valid.groupby("award_type")["is_junior"].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Career age histogram — clean vs flagged
axes[0].hist(valid["career_age"].dropna(), bins=30, color="steelblue", alpha=0.8, label="Valid")
axes[0].hist(
    df[~df["career_age_valid"] & df["career_age"].notna()]["career_age"],
    bins=30, color="tomato", alpha=0.6, label="Flagged"
)
axes[0].axvline(5, color="black", linestyle="--", linewidth=1, label="Junior cutoff (5yr)")
axes[0].set_xlabel("Career Age at Award")
axes[0].set_title("Career Age Distribution")
axes[0].legend()

# Junior/senior bar by conference
conf_grp = valid.groupby(["conference", "is_junior"]).size().unstack(fill_value=0)
conf_grp.columns = ["Senior", "Junior"]
conf_grp.plot(kind="bar", stacked=True, ax=axes[1], color=["steelblue", "orange"])
axes[1].set_title("Junior vs Senior by Conference")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(loc="upper right")

# Median career age first-authors only
fa = valid[valid["author_position"] == "first"]
axes[2].hist(fa["career_age"].dropna(), bins=20, color="seagreen", alpha=0.8)
axes[2].axvline(5, color="black", linestyle="--", linewidth=1)
axes[2].set_xlabel("Career Age at Award")
axes[2].set_title("First Authors Only")

plt.tight_layout()
plt.savefig("../data/figures/37_career_age_eda.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → data/figures/37_career_age_eda.png")

In [ ]:
# First-author summary
print("First-author junior share by conference:")
print(fa.groupby("conference")["is_junior"].mean().round(3))

print("\nMedian career age (valid rows):")
print(valid.groupby(["conference", "author_position"])["career_age"].median().round(1))

print("\nMedian total_works & total_cites: junior vs senior")
print(valid.groupby("is_junior")[["total_works", "total_cites"]].median().round(1))